# 🧩 LoRA & QLoRA — Concepts, Usage Examples, and a Basic Fine-Tuning Project

This notebook is the hands-on companion to **Module 4** of the AI-ML repo. It follows the structure of
**["QLoRA — How to Fine-Tune an LLM on a Single GPU (w/ Python Code)"](https://www.youtube.com/watch?v=XpoKB3usmKc)** by Shaw Talebi — watch that video alongside this notebook for the full intuition.

**Runtime:** In Colab go to `Runtime > Change runtime type > T4 GPU` (or any CUDA GPU with ≥ 8GB VRAM). On an Apple Silicon Mac with no CUDA GPU, use `../mac_mlx_lora/` instead — same idea, MLX backend.

## What you'll do
1. Understand LoRA and QLoRA — the concept and the math
2. See real usage patterns (domain adaptation, persona adapters, multi-tenant serving)
3. Load and inspect the Guanaco instruction dataset
4. Load Mistral-7B-Instruct in 4-bit (the "Q" in QLoRA)
5. Ask it a question **before** fine-tuning
6. Attach a LoRA adapter and fine-tune with `trl.SFTTrainer`
7. Compare the same question **after** fine-tuning — and toggle the adapter on/off live
8. Save the adapter, and optionally merge it into the base model for deployment

## 1. LoRA in one picture

Full fine-tuning updates every weight matrix `W` (shape `d × k`) directly. For a 7B model that means
gradients and optimizer state for all 7 billion parameters — often 4–8x the model size in VRAM.

**LoRA (Low-Rank Adaptation)** freezes `W` completely and instead learns a small *update* to it:

```
h = W·x + (B·A)·x
```

where `B` is `d × r`, `A` is `r × k`, and the rank `r` is tiny (commonly 8–64) compared to `d, k`.
Only `A` and `B` are trainable. For a `4096 × 4096` layer with `r=16`, that's about `131K` trainable
parameters instead of `16.7M` — a **~99% reduction** for that layer, with `W` never moving.

Because `W` is untouched, you can keep one frozen base model in memory and hot-swap different small
`A`/`B` adapter pairs in and out of it — this is the basis for several of the usage patterns below.

## 2. QLoRA = LoRA + 4-bit quantization

QLoRA keeps the LoRA idea above, but also shrinks the *frozen* base model itself so it fits in far
less GPU memory, via four tricks:

1. **4-bit NormalFloat (NF4)** — a 4-bit data type tuned to the roughly-normal distribution of neural
   network weights, more accurate than a plain 4-bit integer at the same size.
2. **Double Quantization** — quantizes the quantization constants themselves for a bit more savings.
3. **Paged Optimizers** — uses unified memory to page optimizer states to CPU RAM instead of crashing
   on a GPU memory spike.
4. **LoRA adapters** — trained in higher precision (bf16/fp16) on top of the frozen 4-bit base;
   gradients flow *through* the quantized weights into the small adapters.

Net effect: a 7B model needing ~28GB in FP32 can be fine-tuned on a **single consumer GPU with
~6–10GB VRAM** — including a free Colab T4 — at a small cost to final quality vs. full fine-tuning.

## 3. Where this is used in practice

- **Domain adaptation** — adapt a general chatbot to legal, medical, or internal-company text without touching the base weights.
- **Persona / tone adapters** — one base model, many swappable "personality" adapters (formal support agent, casual assistant, a specific brand voice). We'll demo the swap mechanism (not train two full personas) later in this notebook.
- **Instruction tuning** — turning a base/instruct model into a better instruction-follower for a specific style — exactly what we do below.
- **Multi-tenant serving** — serve many customers from one loaded base model by hot-swapping a few-MB adapter per request, instead of a full fine-tuned model per customer.
- **Style/format transfer** — e.g. training a model to always reply in a fixed JSON schema, or in a specific reply style (Shaw Talebi's original tutorial: a YouTube-comment responder).

## 4. Setup

In [ ]:
!pip install -q -U "transformers>=4.46" "datasets>=2.20" "accelerate>=1.0" "peft>=0.13" "trl>=1.9" "bitsandbytes>=0.44" "huggingface_hub>=0.25" sentencepiece

In [ ]:
# Mistral-7B-Instruct is not a gated model, so this is usually not needed.
# If you swap in a gated model (e.g. Llama or Gemma) and hit an auth error, uncomment:
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "No GPU found -- in Colab: Runtime > Change runtime type > GPU"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} | compute dtype: {compute_dtype}")

## 5. Dataset: Guanaco (1k subset)

[`mlabonne/guanaco-llama2-1k`](https://huggingface.co/datasets/mlabonne/guanaco-llama2-1k) is a
1,000-example subset of the OpenAssistant **Guanaco** dataset — the same dataset family used in the
original QLoRA paper — pre-formatted into the Llama-2 prompt style:

```
<s>[INST] {instruction} [/INST] {response} </s>
```

It's small on purpose: a full pass over it takes a few minutes on a free Colab T4, which keeps this
a genuinely *basic* project you can iterate on quickly, rather than an overnight training run.

In [ ]:
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
print(dataset)
print("\n--- Example row ---\n")
print(dataset[0]["text"])

## 6. Load the base model in 4-bit (this is the "Q" in QLoRA)

Swap `MODEL_ID` for a smaller model (e.g. `Qwen/Qwen2.5-1.5B-Instruct` or
`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) if you want a faster end-to-end run, or don't have a GPU with
much VRAM. The rest of the notebook works unchanged either way.

In [ ]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print(base_model.get_memory_footprint() / 1e9, "GB on GPU")

## 7. Baseline: how does it answer before fine-tuning?

We capture this answer *now*, before attaching any adapter, so we have a true "before" to compare
against later (once a LoRA adapter is attached, generating from this same model object will run
through the adapter too).

In [ ]:
def ask(model, prompt, max_new_tokens=150):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)


TEST_PROMPT = "In one paragraph, explain what a low-rank matrix decomposition is and why it is useful."
baseline_answer = ask(base_model, TEST_PROMPT)
print(baseline_answer)

## 8. Attach a LoRA adapter

`target_modules="all-linear"` attaches LoRA to every linear layer in the model (attention *and*
MLP projections). It's a convenience shorthand (PEFT ≥ 0.12) that works across model families, so
you don't need to hardcode layer names like `q_proj`/`v_proj` if you swap `MODEL_ID` above.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

## 9. Fine-tune with `SFTTrainer`

This is where the two halves of QLoRA meet: `base_model` is already frozen and 4-bit quantized, and
`peft_config` tells `SFTTrainer` to wrap it with trainable LoRA adapters before training. The dataset
already has a `text` column in the right format, so no `formatting_func` is needed.

In [ ]:
training_args = SFTConfig(
    output_dir="./mistral-guanaco-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    report_to="none",
)

trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
)

trainer.train()

## 10. Save the adapter

Notice how small this is compared to the 7B base model — typically a few tens of MB. That's the
whole point: you're shipping a small delta, not a new copy of the model.

In [ ]:
ADAPTER_DIR = "./mistral-guanaco-lora/final_adapter"
trainer.save_model(ADAPTER_DIR)

import subprocess
print(subprocess.run(["du", "-sh", ADAPTER_DIR], capture_output=True, text=True).stdout)

## 11. Compare: before vs. after — and toggling the adapter on/off live

`trainer.model` is the *same* underlying weights as `base_model`, now with a LoRA adapter attached.
PEFT lets you disable that adapter on the fly with a context manager — no reloading the base model
from disk — which is the mechanism behind the "swap adapters" usage pattern described above.

In [ ]:
tuned_model = trainer.model

print("--- BEFORE (captured earlier, base model) ---")
print(baseline_answer)

print("\n--- AFTER (LoRA adapter enabled) ---")
print(ask(tuned_model, TEST_PROMPT))

print("\n--- Adapter disabled on the SAME loaded model (no reload) ---")
with tuned_model.disable_adapter():
    print(ask(tuned_model, TEST_PROMPT))

> **Honest expectation-setting:** this is a *basic* demo run — 1 epoch over 1,000 examples. It's
> enough to see the training loop work end-to-end and the loss drop, and often enough to nudge
> style/format. It is **not** enough data or compute to expect a dramatic jump in reasoning quality.
> For a real project, expect to iterate on epochs, learning rate, `r`, and dataset size/quality.

## 12. (Optional) Merge the adapter into the base model for deployment

Merging bakes the LoRA update directly into `W` (`W' = W + B·A`), producing one standalone model
that no longer needs PEFT installed to load. Merging requires the base model in full precision
(not 4-bit), so we reload it that way first — only the adapter (a few MB) needs to be re-read from
disk, not retrained.

In [ ]:
merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=compute_dtype, device_map="auto"
)
merged = PeftModel.from_pretrained(merge_base, ADAPTER_DIR)
merged = merged.merge_and_unload()

merged.save_pretrained("./mistral-guanaco-merged")
tokenizer.save_pretrained("./mistral-guanaco-merged")
print("Merged model saved to ./mistral-guanaco-merged -- loadable with plain from_pretrained(), no PEFT needed.")

## 13. Next steps

- **Swap the dataset**: point `load_dataset(...)` at your own domain data (support tickets, docs,
  a specific writing style) to see LoRA's real value — domain adaptation, not just a toy demo.
- **Swap the model**: change `MODEL_ID` to a smaller model for faster iteration, or a bigger one if
  you have the VRAM. `target_modules="all-linear"` means the LoRA config doesn't need to change.
- **Try the Mac-native path**: `../mac_mlx_lora/` runs the same idea with `mlx-lm` on Apple Silicon,
  no CUDA GPU required.
- **Go deeper on the math**: [LoRA paper](https://arxiv.org/abs/2106.09685),
  [QLoRA paper](https://arxiv.org/abs/2305.14314), and the videos linked in the module
  [`README.md`](../README.md).